# Data Ingestion Subsystem

This notebook orchestrates the vectorized data ingestion process, adhering strictly to the design principles of **Hexagonal Architecture (Ports & Adapters)** and **SOLID** paradigms (specifically, Dependency Inversion and Interface Segregation).

## Architectural Design

The orchestrator of this workflow is the `DataIngestionService` class.

- **Source (Reader):** `BloombergDataHandler` implements the `IDataHandler` interface.
- **Destination (Writer):** `HDF5DataHandler` or `ParquetDataHandler` implements the `IDataWriter` interface.

The use of these explicit interfaces decouples the use case from the underlying concrete technologies.

In [1]:
import os
import sys
from pathlib import Path

# Add the project root to sys.path to allow importing from src/ robustly
current_dir = Path(os.path.abspath(''))
project_root = current_dir
while not (project_root / 'src').exists() and project_root.parent != project_root:
    if (project_root / 'Algorithmic_Trading_Backtester' / 'src').exists():
        project_root = project_root / 'Algorithmic_Trading_Backtester'
        break
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.infrastructure.BloombergDataHandler import BloombergDataHandler
from src.infrastructure.DataIngestionService import DataIngestionService
from src.infrastructure.HDF5DataHandler import HDF5DataHandler


## Initialization of Ports and Adapters

The concrete implementations are instantiated and subsequently injected into the domain orchestration service.

In [2]:
# Extraction Parameters
START_DATE = '2020-01-01'
END_DATE = '2026-06-01'
SYMBOLS = ['SPX Index', 'IBEX Index']

# 1. Configure the Reader 
class MockApiConnection:
    pass

try:
    reader = BloombergDataHandler(api_connection=MockApiConnection(), start_date=START_DATE, end_date=END_DATE)
    bloomberg_available = True
except Exception as e:
    print(f"[WARNING] Cannot initialize or connect BloombergDataHandler: {e}")
    bloomberg_available = False
    reader = None

# 2. Configure the Writer
data_path = os.path.join(project_root, 'data', 'historical')
os.makedirs(data_path, exist_ok=True)

hdf5_path = os.path.join(data_path, 'market_data.h5')
writer = HDF5DataHandler(file_path=hdf5_path)

# Alternatively, Parquet can be used:
# parquet_path = os.path.join(data_path, 'parquet_store')
# writer = ParquetDataHandler(dir_path=parquet_path)

# 3. Inject dependencies (DIP)
if bloomberg_available:
    ingestion_service = DataIngestionService(reader=reader, writer=writer)
else:
    ingestion_service = None
    print("Ingestion service is disabled due to missing Bloomberg bindings.")


[WARNING] Cannot initialize or connect BloombergDataHandler: The 'xbbg' and 'blpapi' libraries must be installed (pip install xbbg blpapi) to interact with the Bloomberg Terminal.
Ingestion service is disabled due to missing Bloomberg bindings.


## Execution of the Asymptotically Optimized Ingestion

The service abstracts the Bloomberg data extraction and subsequent persistence into the `market_data.h5` file. This process avoids any explicit iteration over the time-series blocks, ensuring an optimal asymptotic time complexity of $\mathcal{O}(1)$ per matrix operation.

In [3]:
if ingestion_service is not None:
    try:
        print("Starting vectorized ingestion process...")
        ingestion_service.ingest(symbols=SYMBOLS)
        print("Ingestion complete.")
    except Exception as e:
        print(f"Runtime Error during ingestion: {e}")
else:
    print("Skipping execution as Bloomberg bindings are missing.")


Skipping execution as Bloomberg bindings are missing.
